# AI工学101 — 第29回

## ハイパーパラメータ探索を本気でやる：GridSearchCV / RandomizedSearchCV

第28回では、モデルの中身を読むところまで進みました。

今日は、第19回・第23回あたりで触れた**ハイパーパラメータ探索**を、ここで一度きっちり体系化します。

これまで、

```text
モデルを選ぶ
↓
パラメータを設定する
↓
Cross Validation
↓
評価
```

としてきました。

今日はその、

> **「パラメータをどう選ぶのか？」**

を実験として設計します。

---

# 🎯 今日のゴール

* ハイパーパラメータと学習パラメータの違いを説明できる
* `GridSearchCV` を使える
* `RandomizedSearchCV` を使える
* `best_params_` と `best_score_` を解釈できる
* CV用データと最終testデータを区別できる
* 探索しすぎによる過学習を理解する
* 「探索空間そのものを設計する」という考え方を身につける

---

# 📖 講義：約20分

## 1. パラメータには2種類ある

まずここを整理します。

### 学習によって決まるもの

例えばLinear Regressionなら、

```text
W
b
```

など。

これは、

```python
model.fit(X, y)
```

によってデータから学習されます。

これを一般に**パラメータ**と呼びます。

---

### 人間が設定するもの

例えばRandom Forestなら、

```python
n_estimators=100
max_depth=5
```

など。

これらは `fit()` が勝手に決めるのではなく、

**人間が設定するハイパーパラメータ**です。

---

# 🧠 2. なぜ探索するのか？

例えばRandom Forestで、

```text
max_depth = 1
```

なら単純すぎるかもしれない。

```text
max_depth = None
```

なら複雑すぎるかもしれない。

じゃあ、

```text
2？
3？
5？
10？
```

と試してみよう。

これがハイパーパラメータ探索です。

---

# 📖 3. Grid Search

最も直感的なのが、

**Grid Search**

です。

例えば、

```python
param_grid = {
    "max_depth": [2, 3, 5],
    "n_estimators": [50, 100, 200]
}
```

なら、

```text
3 × 3 = 9通り
```

全部試します。

```text
depth=2, trees=50
depth=2, trees=100
depth=2, trees=200

depth=3, trees=50
...
```

という感じ。

---

# 💻 実習1：Random ForestのGrid Search

まずデータ。

```python
from sklearn.datasets import load_breast_cancer

data = load_breast_cancer()

X = data.data
y = data.target
```

train/test分割。

```python
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)
```

---

# 💻 実習2：ベースモデル

```python
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    random_state=42
)
```

探索空間。

```python
param_grid = {
    "n_estimators": [
        50,
        100,
        200
    ],
    "max_depth": [
        2,
        5,
        10,
        None
    ]
}
```

組み合わせは、

```text
3 × 4 = 12
```

通り。

---

# 💻 実習3：GridSearchCV

```python
from sklearn.model_selection import GridSearchCV

grid = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)
```

学習。

```python
grid.fit(
    X_train,
    y_train
)
```

これで、

```text
12設定
×
5-fold CV
```

が実行されます。

つまり概念的には、

```text
60回程度のモデル評価
```

をしています。

---

# 💻 実習4：最適パラメータ

```python
print(
    grid.best_params_
)
```

例えば、

```python
{
    "max_depth": 5,
    "n_estimators": 100
}
```

のような結果になります。

これは、

> **CV上で最も良かった設定**

です。

---

# 💻 実習5：best_score_

```python
print(
    grid.best_score_
)
```

これは、

> **CVで得られた最良設定の平均スコア**

です。

ここで注意。

これは、

**test setの性能ではありません。**

---

# 💻 実習6：最終test評価

最良モデル。

```python
best_model = grid.best_estimator_
```

testデータで評価。

```python
test_score = best_model.score(
    X_test,
    y_test
)

print(
    "Test accuracy:",
    test_score
)
```

ここで初めて、

> **未知データに対する最終性能**

を確認します。

---

# 🚨 4. CVとtestを混ぜない

今日の最重要ポイント。

正しい流れは、

```text
全データ
   ↓
train / test
   ↓
train
 ┌───────────────┐
 ↓               ↓
CV             GridSearch
 ↓               ↓
ハイパーパラメータ選択
       ↓
   最良モデル
       ↓
     test
       ↓
最終性能
```

です。

test setは、

> **最後までなるべく触らない**

。

なぜなら、testを見ながら、

```text
この設定が良さそう
↓
じゃあ別の設定
↓
またtestを見る
```

を繰り返すと、

**testデータに対しても最適化してしまう**

からです。

---

# 🧠 5. 「test setへの過学習」

ここはかなり重要な考え方。

普通の過学習は、

```text
モデル
 ↓
trainデータに合わせすぎる
```

でした。

でも実験を何度も繰り返して、

```text
testを見る
↓
設定を変える
↓
またtestを見る
↓
設定を変える
```

とやると、

```text
研究者
 ↓
testデータに合わせて意思決定
```

してしまいます。

つまり、

> **人間がtest setに過学習する**

ような状態になります。

だから、

```text
CV
```

をハイパーパラメータ選択に使い、

```text
test
```

を最後の確認に残します。

---

# 💻 実習7：GridSearchCVの結果を見る

```python
results = grid.cv_results_
```

例えば、

```python
print(
    results["params"]
)
```

```python
print(
    results["mean_test_score"]
)
```

を見ることができます。

さらに、

```python
for params, score in zip(
    results["params"],
    results["mean_test_score"]
):

    print(
        params,
        score
    )
```

とすると、

```text
どの設定
↓
どのCVスコア
```

だったか確認できます。

---

# 💻 実習8：標準偏差も見る

```python
print(
    results["std_test_score"]
)
```

平均だけではなく、

```text
CV平均
CV標準偏差
```

を見ることで、

> **その設定の性能がFoldによってどれくらい揺れたか**

も確認できます。

---

# 📖 6. Grid Searchの弱点

Grid Searchは分かりやすい。

でも、

```text
パラメータが増える
```

と組み合わせ数が爆発します。

例えば、

```text
パラメータA → 10候補
パラメータB → 10候補
パラメータC → 10候補
パラメータD → 10候補
```

なら、

$$
10^4=10000
$$

通り。

さらに、

```text
5-fold CV
```

なら、

```text
50,000回
```

規模の学習・評価が発生し得ます。

これはかなり重い。

---

# 📖 7. Randomized Search

そこで、

**RandomizedSearchCV**

です。

Grid Searchは、

```text
候補を全部試す
```

でした。

Randomized Searchは、

> **探索空間からランダムに設定をサンプリングする**

という方法です。

---

# 💻 実習9：RandomizedSearchCV

```python
from sklearn.model_selection import RandomizedSearchCV
```

探索空間。

```python
param_distributions = {
    "n_estimators": [
        50,
        100,
        200,
        300,
        500
    ],
    "max_depth": [
        2,
        3,
        5,
        10,
        None
    ],
    "min_samples_split": [
        2,
        5,
        10
    ]
}
```

モデル。

```python
model = RandomForestClassifier(
    random_state=42
)
```

Randomized Search。

```python
search = RandomizedSearchCV(
    estimator=model,
    param_distributions=param_distributions,
    n_iter=10,
    cv=5,
    scoring="accuracy",
    random_state=42,
    n_jobs=-1
)
```

学習。

```python
search.fit(
    X_train,
    y_train
)
```

---

# 🧠 `n_iter`

ここ重要。

```python
n_iter=10
```

なら、

> **ランダムに選んだ10通りだけ試す**

ということ。

つまり、

```text
候補が大量
       ↓
全部は無理
       ↓
ランダムに10個選ぶ
       ↓
CVで評価
```

です。

---

# 💻 実習10：Grid vs Randomized

比較。

```python
print(
    "Grid:",
    grid.best_score_
)

print(
    "Random:",
    search.best_score_
)
```

そして、

```python
print(
    "Grid params:",
    grid.best_params_
)

print(
    "Random params:",
    search.best_params_
)
```

を見ます。

必ずしも、

```text
Grid Search > Random Search
```

とは限りません。

探索回数が同じでも、ランダム探索のほうが良い領域を偶然見つけることもあります。

---

# 🧠 8. 探索空間をどう作るか

ここが今日の「工学」部分です。

例えば、

```python
"max_depth": [
    1,
    2,
    3,
    4,
    5,
    6,
    7,
    8,
    9,
    10
]
```

とするのも一つ。

でも、

```text
深さ1〜10
```

を全部試す必要があるでしょうか？

過去の実験で、

```text
1 → underfitting
2 → 良い
3 → 良い
4 → 良い
5 → 過学習気味
```

と分かっていたなら、

```text
2〜4
```

を重点的に調べるほうが合理的かもしれません。

つまり、

> **ハイパーパラメータ探索は「全部試すゲーム」ではない。**

です。

過去の実験結果やモデルの性質を使って、

**探索空間そのものを設計する**

のが重要です。

---

# 💻 実習11：複数指標で評価する

第24回に戻ります。

例えば、

```python
scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc"
}
```

とできます。

`GridSearchCV` では、

```python
grid = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    scoring=scoring,
    refit="f1",
    cv=5,
    n_jobs=-1
)
```

のように、

**複数指標を記録しつつ、最終的にどの指標を基準に最良モデルを選ぶか**

を指定できます。

ここでは、

```python
refit="f1"
```

なので、

> **F1が最も良かったモデルを最終モデルとして再学習**

します。

---

# 🧠 これはかなり重要

つまり、

```text
モデル探索
```

だけではなく、

```text
何を最適化するか
```

まで指定しています。

第24・25回の、

```text
Accuracy
Precision
Recall
F1
ROC-AUC
```

がここで実際のモデル選択につながりました。

---

# 💻 実習12：Gradient Boostingを探索する

第23回に戻ります。

```python
from sklearn.ensemble import GradientBoostingClassifier

model = GradientBoostingClassifier(
    random_state=42
)
```

探索空間。

```python
param_grid = {
    "n_estimators": [
        50,
        100,
        200
    ],
    "learning_rate": [
        0.05,
        0.1,
        0.2
    ],
    "max_depth": [
        1,
        2,
        3
    ]
}
```

Grid Search。

```python
grid_gb = GridSearchCV(
    model,
    param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)
```

```python
grid_gb.fit(
    X_train,
    y_train
)
```

最適値。

```python
print(
    grid_gb.best_params_
)
```

---

# 💻 実習13：最終test評価

```python
best_gb = (
    grid_gb.best_estimator_
)
```

```python
print(
    "CV:",
    grid_gb.best_score_
)

print(
    "Test:",
    best_gb.score(
        X_test,
        y_test
    )
)
```

ここで、

```text
CV
vs
Test
```

を比較します。

---

# 👾 ボス戦：モデル選択の最終実験

ここまで学んだものを統合します。

候補を、

```text
Logistic Regression
Random Forest
Gradient Boosting
```

とします。

それぞれについて、

1. train/testを分割
2. trainだけでGridSearchCV
3. 5-fold CV
4. Accuracy / F1 / ROC-AUCを記録
5. 最良モデルを選択
6. 最後にtest setを1回だけ評価

してください。

---

## 最終的に作る表

```text
Model
CV Accuracy
CV F1
CV ROC-AUC
Test Accuracy
Test F1
Test ROC-AUC
```

を比較します。

そして、

> **どのモデルを採用するか**

を決めます。

ただし、

```text
Accuracyが一番高い
```

だけでは不十分。

第24回・25回で学んだ、

```text
問題の目的
↓
失敗コスト
↓
評価指標
```

まで含めて判断します。

---

# 🧠 ボス戦のもう一つのポイント

例えば、

```text
Model A
CV F1 = 0.97
Test F1 = 0.91

Model B
CV F1 = 0.95
Test F1 = 0.94
```

だったら、

単純に、

```text
CV最高のA
```

を採用するのが必ずしも良いとは限りません。

Aは、

```text
CVでは良かった
↓
testで大きく落ちた
```

ということだからです。

ここから、

> **CVスコアだけでなく、汎化性能と安定性を見る**

という視点につながります。

---

# 🌱 今日のまとめ

今日の核心は、

> **ハイパーパラメータ探索は、単なる「最適値探し」ではなく、実験設計である。**

ということ。

流れとしては、

```text
train/test split
      ↓
    train
      ↓
  CV + Search
      ↓
best parameters
      ↓
best model
      ↓
    test
      ↓
final performance
```

です。

そして、

```text
GridSearchCV
```

は、

> **指定した候補を網羅的に調べる**

。

```text
RandomizedSearchCV
```

は、

> **広い探索空間から限られた回数だけランダムに調べる**

。

さらに、

```text
探索空間
評価指標
CV
refit
計算コスト
```

まで含めて設計する。

ここまで来ると、かなり「AIを作る」側の思考になっています。

---

# 🧭 AI工学101・現在地

scikit-learn編の基礎が、ほぼ一本につながりました。

```text
データ
 ↓
前処理
 ↓
特徴量
 ↓
特徴量選択 / PCA
 ↓
モデル候補
 ↓
学習
 ↓
予測確率
 ↓
Threshold
 ↓
評価指標
 ↓
Cross Validation
 ↓
Hyperparameter Search
 ↓
モデル解釈
 ↓
最終test
```

つまり、

**「モデルを使える」から「機械学習実験を設計できる」へ進んでいる。**

ここはAI工学101のかなり大きな節目です。

---

# 🔜 第30回

## 学習曲線・検証曲線・バイアス/バリアンス：モデルが「なぜ」うまくいかないのか

次回はいったんハイパーパラメータの数字から離れて、

> **「このモデル、そもそも何が原因で性能が悪いんだ？」**

を診断します。

扱うのは、

* Learning Curve
* Validation Curve
* Underfitting
* Overfitting
* Bias / Variance
* データ量と性能の関係
* モデルの複雑さと性能の関係
* 「もっとデータが必要なのか？」「もっと複雑なモデルが必要なのか？」を判断する方法

です。

ここで第20回から続いてきた**汎化・過学習**を、グラフを使って「診断する」段階へ進みます。
